In [0]:
from pyspark.sql.functions import *

bronze_path = "/Volumes/workspace/default/mis_datasets_mhealth/bronze"

df = spark.read.format("delta").load(bronze_path)

display(df)

In [0]:
df.count()

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(null_counts)

In [0]:
df_silver = df.dropna()

In [0]:
df_silver = (
    df_silver
    .withColumn("subject_id", col("subject_id").cast("int"))
    .withColumn("activity_label", col("activity_label").cast("int"))
)

In [0]:
activity_mapping = {
    0: "Null",
    1: "Standing",
    2: "Sitting",
    3: "Lying Down",
    4: "Walking",
    5: "Climbing Stairs",
    6: "Waist Bends",
    7: "Arm Elevation",
    8: "Knee Bends",
    9: "Cycling",
    10: "Jogging",
    11: "Running",
    12: "Jumping"
}

In [0]:
mapping_expr = create_map(
    [lit(x) for pair in activity_mapping.items() for x in pair]
)

df_silver = df_silver.withColumn(
    "activity_name",
    mapping_expr[col("activity_label")]
)

In [0]:
display(df_silver)

In [0]:
df_silver = df_silver.withColumn(
    "ecg_signal_avg",
    (col("ecg_lead_1") + col("ecg_lead_2")) / 2
)

In [0]:
silver_path = "/Volumes/workspace/default/mis_datasets_mhealth/silver"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

In [0]:
silver_df = spark.read.format("delta").load(silver_path)

display(silver_df)

In [0]:
df_silver.select("activity_label").distinct().display()

In [0]:
df_silver = df_silver.drop("activity_name")

In [0]:
from pyspark.sql.functions import *

df_silver = df_silver.withColumn(
    "activity_name",
    when(col("activity_label") == 0, "Null")
    .when(col("activity_label") == 1, "Standing")
    .when(col("activity_label") == 2, "Sitting")
    .when(col("activity_label") == 3, "Lying Down")
    .when(col("activity_label") == 4, "Walking")
    .when(col("activity_label") == 5, "Climbing Stairs")
    .when(col("activity_label") == 6, "Waist Bends")
    .when(col("activity_label") == 7, "Arm Elevation")
    .when(col("activity_label") == 8, "Knee Bends")
    .when(col("activity_label") == 9, "Cycling")
    .when(col("activity_label") == 10, "Jogging")
    .when(col("activity_label") == 11, "Running")
    .when(col("activity_label") == 12, "Jumping")
)

In [0]:
display(
    df_silver.select("activity_label", "activity_name").distinct().orderBy("activity_label")
)

In [0]:
df_silver = df_silver.withColumn(
    "ecg_signal_avg",
    (col("ecg_lead_1") + col("ecg_lead_2")) / 2
)

In [0]:
display(df_silver)

In [0]:
silver_path = "/Volumes/workspace/default/mis_datasets_mhealth/silver"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

In [0]:
silver_df = spark.read.format("delta").load(silver_path)

display(silver_df)